# DuyAI: Azərbaycan dilli zəng anlayışı modelinin train edilməsi

**Kaggle ayarları:** Accelerator → **GPU T4 x2** (və ya P100), Internet → **On**.

Əvvəlcə `training` qovluğunu Kaggle Dataset kimi yükləyin (adı: `duyai-training`). İçində bunlar olmalıdır: `common.py`, `train.py`, `predict.py`, `data/train.jsonl`, `data/eval_inputs.jsonl`.

In [ ]:
!pip install -q -U peft bitsandbytes accelerate datasets transformers
# Kaggle-ın köhnə torchao-su peft ilə uyğunsuzdur və bizə lazım deyil
!pip uninstall -y -q torchao

In [ ]:
import shutil, os, glob, subprocess
src = glob.glob('/kaggle/input/**/common.py', recursive=True)[0].rsplit('/', 1)[0]
shutil.copytree(src, '/kaggle/working/training', dirs_exist_ok=True)
os.chdir('/kaggle/working/training')
print(os.listdir('.'), os.listdir('data'))

def run(cmd):
    """Runs a step and stops the notebook if it fails ('!cmd' would silently continue)."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-6000:])
    if r.returncode:
        print(r.stderr[-6000:])
        raise RuntimeError(f'FAILED: {cmd}')

## 1. Baza model (train-siz): müqayisə üçün

In [ ]:
run('python predict.py --out predictions_base.jsonl')

## 2. Train (QLoRA, ~20–60 dəq. T4-də, nümunə sayından asılıdır)

In [ ]:
# Kiçik dataset üçün daha çox epoch: ~25 nümunədə 3 epoch cəmi bir neçə addımdır
n = sum(1 for l in open('data/train.jsonl', encoding='utf-8') if l.strip())
epochs = 3 if n >= 300 else 8
print(n, 'nümunə ->', epochs, 'epoch')
run(f'python train.py --data data/train.jsonl --out duyai-lora --epochs {epochs}')

## 3. Train olunmuş model: test toplusunda proqnoz

In [ ]:
run('python predict.py --adapter duyai-lora --out predictions_finetuned.jsonl')

## 4. Nəticələri yükləyin

`predictions_base.jsonl` və `predictions_finetuned.jsonl` fayllarını (Output bölməsindən) layihənin `training/` qovluğuna köçürün və kompüterdə işlədin:

```
node eval/run.mjs --predictions training/predictions_base.jsonl --name "Qwen2.5-1.5B (baza)"
node eval/run.mjs --predictions training/predictions_finetuned.jsonl --name "Qwen2.5-1.5B + LoRA"
```

In [ ]:
shutil.make_archive('/kaggle/working/duyai_outputs', 'zip', '.', '.')
print('Hazırdır: /kaggle/working/duyai_outputs.zip')